# 파이썬(Python)의 데이터 유형 이해

효과적인 데이터 기반 과학 및 계산을 위해서는 데이터가 저장되고 조작되는 방식을 이해해야 합니다.
이번 장에서는 파이썬(Python) 언어 자체에서 데이터 배열을 처리하는 방법과 NumPy가 이를 개선하는 방법을 간략하게 설명하고 대조합니다.
이 차이점을 이해하는 것은 이 책의 나머지 부분에 걸쳐 많은 내용을 이해하는 데 필수적입니다.

파이썬(Python) 사용자는 종종 사용하기 쉽다는 점에 매력을 느낍니다. 그 중 하나는 동적 타이핑입니다.
C나 Java와 같은 정적으로 유형이 지정된 언어에서는 각 변수를 명시적으로 선언해야 하지만 파이썬(Python)과 같은 동적으로 유형이 지정된 언어는 이 사양을 건너뜁니다. 예를 들어 C에서는 다음과 같이 특정 작업을 지정할 수 있습니다.

``C
/* C 코드 */
정수 결과 = 0;
for(int i=0; i<100; i++){
결과 += i;
}
````

파이썬(Python)에서는 동일한 작업을 다음과 같이 작성할 수 있습니다.

``파이썬
# 파이썬 코드
결과 = 0
범위(100)에 있는 i의 경우:
결과 += 나
````

한 가지 주요 차이점에 주목하세요. C에서는 각 변수의 데이터 유형이 명시적으로 선언되는 반면 파이썬(Python)에서는 유형이 동적으로 유추됩니다. 이는 예를 들어 모든 종류의 데이터를 모든 변수에 할당할 수 있음을 의미합니다.

``파이썬
# 파이썬 코드
엑스 = 4
x = "4"
````

여기서는 `x`의 내용을 정수에서 문자열로 전환했습니다. C에서도 동일한 문제가 (컴파일러 설정에 따라) 컴파일 오류나 기타 의도하지 않은 결과를 초래할 수 있습니다.

``C
/* C 코드 */
정수 x = 4;
x = "4";  // 실패
````

이러한 종류의 유연성은 파이썬(Python) 및 기타 동적으로 유형이 지정된 언어를 편리하고 사용하기 쉽게 만드는 요소 중 하나입니다.
이것이 *어떻게* 작동하는지 이해하는 것은 파이썬(Python)을 사용하여 효율적이고 효과적으로 데이터를 분석하는 학습의 중요한 부분입니다.
그러나 이러한 유형의 유연성이 가리키는 점은 파이썬(Python) 변수가 단순한 값 이상이라는 사실입니다. 또한 값의 *유형*에 대한 추가 정보도 포함합니다. 이에 대해서는 다음 섹션에서 더 자세히 살펴보겠습니다.

## 파이썬 정수는 단순한 정수 그 이상입니다

표준 파이썬(Python) 구현은 C로 작성되었습니다.
이는 모든 파이썬(Python) 객체가 그 값뿐만 아니라 다른 정보도 포함하는 영리하게 위장된 C 구조라는 것을 의미합니다. 예를 들어 파이썬(Python)에서 `x = 10000`과 같은 정수를 정의할 때 `x`는 단순한 "원시" 정수가 아닙니다. 실제로는 여러 값을 포함하는 복합 C 구조에 대한 포인터입니다.
파이썬(Python) 3.10 소스 코드를 살펴보면 정수(long) 유형 정의가 사실상 다음과 같다는 것을 알 수 있습니다(C 매크로가 확장되면).

``C
구조체 _longobject {
긴 ob_refcnt;
PyTypeObject *ob_type;
size_t ob_size;
긴 ob_digit[1];
};
````

파이썬(Python) 3.10의 단일 정수에는 실제로 네 부분이 포함됩니다.

- 파이썬(Python)이 자동으로 메모리 할당 및 할당 해제를 처리하는 데 도움이 되는 참조 카운트 `ob_refcnt`
- 변수의 유형을 인코딩하는 `ob_type`
- 다음 데이터 멤버의 크기를 지정하는 'ob_size'
- 파이썬(Python) 변수가 나타낼 것으로 예상되는 실제 정수 값을 포함하는 `ob_digit`

이는 다음 그림에 설명된 것처럼 C와 같은 컴파일된 언어에 비해 파이썬(Python)에서 정수를 저장하는 데 약간의 오버헤드가 있음을 의미합니다.

![Integer Memory Layout](images/cint_vs_pyint.png)

여기서 `PyObject_HEAD`는 참조 횟수, 유형 코드 및 앞서 언급한 기타 부분을 포함하는 구조의 일부입니다.

여기서 차이점을 주목하세요. C 정수는 본질적으로 바이트가 정수 값을 인코딩하는 메모리 위치에 대한 레이블입니다.
파이썬(Python) 정수는 정수 값을 포함하는 바이트를 포함하여 모든 파이썬(Python) 개체 정보를 포함하는 메모리 위치에 대한 포인터입니다.
파이썬(Python) 정수 구조의 이 추가 정보를 통해 파이썬(Python)을 매우 자유롭고 동적으로 코딩할 수 있습니다.
그러나 파이썬(Python) 유형의 이 모든 추가 정보에는 비용이 들며, 이는 이러한 객체 중 다수를 결합하는 구조에서 특히 분명해집니다.

## 파이썬(Python) 목록은 단순한 목록 그 이상입니다.

이제 많은 파이썬(Python) 개체를 포함하는 파이썬(Python) 데이터 구조를 사용할 때 어떤 일이 발생하는지 살펴보겠습니다.
파이썬(Python)의 표준 변경 가능한 다중 요소 컨테이너는 목록입니다.
다음과 같이 정수 목록을 만들 수 있습니다.

In [1]:
L = list(range(10))
L

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [2]:
type(L[0])

int

또는 마찬가지로 문자열 목록은 다음과 같습니다.

In [3]:
L2 = [str(c) for c in L]
L2

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

In [4]:
type(L2[0])

str

파이썬(Python)의 동적 타이핑 덕분에 이질적인 목록을 만들 수도 있습니다.

In [5]:
L3 = [True, "2", 3.0, 4]
[type(item) for item in L3]

[bool, str, float, int]

그러나 이러한 유연성에는 비용이 따릅니다. 이러한 유연한 유형을 허용하려면 목록의 각 항목에 고유한 유형, 참조 횟수 및 기타 정보가 포함되어야 합니다. 즉, 각 항목은 완전한 파이썬(Python) 객체입니다.
모든 변수가 동일한 유형인 특별한 경우에는 이 정보의 대부분이 중복되므로 고정 유형 배열에 데이터를 저장하는 것이 훨씬 더 효율적일 수 있습니다.
동적 유형 목록과 고정 유형(NumPy 스타일) 배열의 차이점은 다음 그림에 나와 있습니다.

![Array Memory Layout](images/array_vs_list.png)

구현 수준에서 배열에는 기본적으로 하나의 연속된 데이터 블록에 대한 단일 포인터가 포함됩니다.
반면 파이썬(Python) 목록에는 포인터 블록에 대한 포인터가 포함되어 있으며, 각 포인터는 앞서 본 파이썬(Python) 정수와 같은 전체 파이썬(Python) 개체를 가리킵니다.
다시 말하지만 목록의 장점은 유연성입니다. 각 목록 요소는 데이터와 유형 정보를 모두 포함하는 전체 구조이므로 목록은 원하는 유형의 데이터로 채워질 수 있습니다.
고정형 NumPy 스타일 배열에는 이러한 유연성이 부족하지만 데이터를 저장하고 조작하는 데 훨씬 더 효율적입니다.

## 파이썬(Python)의 고정 유형 배열

파이썬(Python)은 효율적인 고정 유형 데이터 버퍼에 데이터를 저장하기 위한 여러 가지 옵션을 제공합니다.
내장 `배열` 모듈(파이썬(Python) 3.3부터 사용 가능)을 사용하여 균일한 유형의 조밀한 배열을 만들 수 있습니다.

In [6]:
import array
L = list(range(10))
A = array.array('i', L)
A

array('i', [0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

여기서 'i'는 내용이 정수임을 나타내는 타입 코드입니다.

그러나 훨씬 더 유용한 것은 NumPy 패키지의 `ndarray` 개체입니다.
파이썬(Python)의 '배열' 객체가 배열 기반 데이터의 효율적인 저장을 제공하는 반면, NumPy는 해당 데이터에 대한 효율적인 *작업*을 추가합니다.
우리는 이후 장에서 이러한 작업을 탐구할 것입니다. 다음으로 NumPy 배열을 만드는 몇 가지 다른 방법을 보여 드리겠습니다.

## 파이썬(Python) 목록에서 배열 만들기

`np`라는 별칭으로 표준 NumPy 가져오기부터 시작하겠습니다.

In [ ]:
import numpy as np

이제 `np.array`를 사용하여 파이썬(Python) 목록에서 배열을 만들 수 있습니다.

In [8]:
# Integer array
np.array([1, 4, 2, 5, 3])

array([1, 4, 2, 5, 3])

파이썬(Python) 목록과 달리 NumPy 배열은 동일한 유형의 데이터만 포함할 수 있다는 점을 기억하세요.
유형이 일치하지 않으면 NumPy는 유형 승격 규칙에 따라 유형을 업캐스트합니다. 여기서 정수는 부동 소수점으로 업캐스트됩니다.

In [9]:
np.array([3.14, 4, 2, 3])

array([3.14, 4.  , 2.  , 3.  ])

결과 배열의 데이터 유형을 명시적으로 설정하려면 `dtype` 키워드를 사용할 수 있습니다.

In [10]:
np.array([1, 2, 3, 4], dtype=np.float32)

array([1., 2., 3., 4.], dtype=float32)

마지막으로 항상 1차원 시퀀스인 파이썬(Python) 목록과 달리 NumPy 배열은 다차원일 수 있습니다. 목록 목록을 사용하여 다차원 배열을 초기화하는 한 가지 방법은 다음과 같습니다.

In [11]:
# Nested lists result in multidimensional arrays
np.array([range(i, i + 3) for i in [2, 4, 6]])

array([[2, 3, 4],
       [4, 5, 6],
       [6, 7, 8]])

내부 목록은 결과 2차원 배열의 행으로 처리됩니다.

## 처음부터 배열 만들기

특히 대규모 배열의 경우 NumPy에 내장된 루틴을 사용하여 처음부터 배열을 만드는 것이 더 효율적입니다.
다음은 몇 가지 예입니다.

In [12]:
# Create a length-10 integer array filled with 0s
np.zeros(10, dtype=int)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [13]:
# Create a 3x5 floating-point array filled with 1s
np.ones((3, 5), dtype=float)

array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.]])

In [14]:
# Create a 3x5 array filled with 3.14
np.full((3, 5), 3.14)

array([[3.14, 3.14, 3.14, 3.14, 3.14],
       [3.14, 3.14, 3.14, 3.14, 3.14],
       [3.14, 3.14, 3.14, 3.14, 3.14]])

In [15]:
# Create an array filled with a linear sequence
# starting at 0, ending at 20, stepping by 2
# (this is similar to the built-in range function)
np.arange(0, 20, 2)

array([ 0,  2,  4,  6,  8, 10, 12, 14, 16, 18])

In [16]:
# Create an array of five values evenly spaced between 0 and 1
np.linspace(0, 1, 5)

array([0.  , 0.25, 0.5 , 0.75, 1.  ])

In [17]:
# Create a 3x3 array of uniformly distributed
# pseudorandom values between 0 and 1
np.random.random((3, 3))

array([[0.09610171, 0.88193001, 0.70548015],
       [0.35885395, 0.91670468, 0.8721031 ],
       [0.73237865, 0.09708562, 0.52506779]])

In [18]:
# Create a 3x3 array of normally distributed pseudorandom
# values with mean 0 and standard deviation 1
np.random.normal(0, 1, (3, 3))

array([[-0.46652655, -0.59158776, -1.05392451],
       [-1.72634268,  0.03194069, -0.51048869],
       [ 1.41240208,  1.77734462, -0.43820037]])

In [19]:
# Create a 3x3 array of pseudorandom integers in the interval [0, 10)
np.random.randint(0, 10, (3, 3))

array([[4, 3, 8],
       [6, 5, 0],
       [1, 1, 4]])

In [20]:
# Create a 3x3 identity matrix
np.eye(3)

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

In [21]:
# Create an uninitialized array of three integers; the values will be
# whatever happens to already exist at that memory location
np.empty(3)

array([1., 1., 1.])

## NumPy 표준 데이터 유형

NumPy 배열에는 단일 유형의 값이 포함되어 있으므로 해당 유형과 제한 사항에 대해 자세히 아는 것이 중요합니다.
NumPy는 C로 구축되었기 때문에 C, Fortran 및 기타 관련 언어 사용자에게 유형이 익숙할 것입니다.

표준 NumPy 데이터 유형은 다음 표에 나열되어 있습니다.
배열을 구성할 때 문자열을 사용하여 지정할 수 있습니다.

``파이썬
np.zeros(10, dtype='int16')
````

또는 연관된 NumPy 객체를 사용합니다:

``파이썬
np.zeros(10, dtype=np.int16)
````

| 데이터 유형 | 설명 |
|-------------|-------------|
| `부울_` | 바이트로 저장된 부울(True 또는 False) |
| `int_` | 기본 정수 유형(C `long`과 동일, 일반적으로 `int64` 또는 `int32`)|
| `intc` | C `int`와 동일합니다(보통 `int32` 또는 `int64`)|
| `intp` | 인덱싱에 사용되는 정수(C `ssize_t`와 동일, 일반적으로 `int32` 또는 `int64`)|
| `int8` | 바이트(-128~127)|
| `int16` | 정수(–32768 ~ 32767)|
| `int32` | 정수(–2147483648 ~ 2147483647)|
| `int64` | 정수(–9223372036854775808 ~ 9223372036854775807)|
| `uint8` | 부호 없는 정수(0~255)|
| `uint16` | 부호 없는 정수(0~65535)|
| `uint32` | 부호 없는 정수(0~4294967295)|
| `uint64` | 부호 없는 정수(0~18446744073709551615)|
| `float_` | `float64`의 약어|
| 'float16' | 반정밀도 부동 소수점: 부호 비트, 지수 5비트, 가수 10비트|
| 'float32' | 단정밀도 부동 소수점: 부호 비트, 8비트 지수, 23비트 가수|
| 'float64' | 배정밀도 부동 소수점: 부호 비트, 11비트 지수, 52비트 가수|
| `복잡한_` | `complex128`의 약어|
| '복잡한64' | 두 개의 32비트 부동소수점으로 표현되는 복소수|
| `complex128`| 두 개의 64비트 부동소수점으로 표현되는 복소수|

빅 엔디안 또는 리틀 엔디안 숫자 지정과 같은 고급 유형 지정이 가능합니다. 자세한 내용은 [NumPy 설명서](http://numpy.org/)를 참조하세요.
NumPy는 또한 [구조화된 데이터: NumPy의 구조화된 배열](02.09-Structured-Data-NumPy.ipynb)에서 다룰 복합 데이터 유형을 지원합니다.